# Word2Vec: a hands-on implementation

**Track:** Natural language processing · **Guide:** [`030-word2vec.md`](../guides/concepts/030-word2vec.md)

Word2Vec learns dense word vectors by predicting nearby context words or a target word from its context.

> **Intuition:** Words become coordinates based on the neighborhoods they tend to appear with.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Word2Vec]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Inspect learned neighbors for domain terms and remember that similarity reflects training data, not verified meaning.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Build skip-gram training pairs

Word2Vec skip-gram learns to predict nearby context tokens from a center word; this tiny corpus keeps training inspectable.


In [ ]:
import numpy as np

corpus = [
    "battery", "lasts", "long", "battery", "charges",
    "quickly", "charger", "powers", "battery",
]
vocabulary = sorted(set(corpus))
word_id = {word: index for index, word in enumerate(vocabulary)}
pairs = []
for position, word in enumerate(corpus):
    for neighbor in range(max(0, position - 2), min(len(corpus), position + 3)):
        if neighbor != position:
            pairs.append((word_id[word], word_id[corpus[neighbor]]))
print("vocabulary:", vocabulary, "training pairs:", len(pairs))

## 2. Train a small negative-sampling model

The logistic objective raises scores for observed center-context pairs and lowers scores for sampled noise pairs.


In [ ]:
rng = np.random.default_rng(11)
dimension = 8
input_vectors = rng.normal(0, 0.1, (len(vocabulary), dimension))
context_vectors = rng.normal(0, 0.1, (len(vocabulary), dimension))

def update(center, context, label, rate=0.04):
    source = input_vectors[center].copy()
    target_vector = context_vectors[context].copy()
    score = float(source @ target_vector)
    probability = 1 / (1 + np.exp(-np.clip(score, -20, 20)))
    gradient = rate * (label - probability)
    input_vectors[center] += gradient * target_vector
    context_vectors[context] += gradient * source

for _epoch in range(80):
    for center, context in pairs:
        update(center, context, 1)
        for _ in range(2):
            negative = int(rng.integers(len(vocabulary)))
            if negative != context:
                update(center, negative, 0)

## 3. Inspect nearest vector neighbors

These vectors are trained on a tiny corpus and are unstable; real Word2Vec needs a larger, representative corpus and evaluation.


In [ ]:
word = "battery"
vector = input_vectors[word_id[word]]
neighbors = []
for candidate in vocabulary:
    if candidate == word:
        continue
    other = input_vectors[word_id[candidate]]
    similarity = float(vector @ other / (np.linalg.norm(vector) * np.linalg.norm(other) + 1e-9))
    neighbors.append((candidate, similarity))
print(sorted(neighbors, key=lambda pair: -pair[1])[:3])

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/030-word2vec.md) and the
[learning roadmap](../guides/ROADMAP.md).
